# Getting started: reading Indian Railways running data

This dataset is one CSV per train, named by its five digit train number, with the
same 24 columns in every file. One row is **one train, at one station, on one day**.

This notebook covers the three things that will otherwise bite you:

1. Loading a useful subset instead of all 33 GB.
2. The `reported` column, and why every delay number is wrong without it.
3. Delays are never negative, so punctuality here is a lower bound.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Kaggle mounts a dataset under /kaggle/input/<slug>, but the CSVs may sit in
# a subfolder inside it. Find whichever directory actually holds them rather
# than assuming, so this runs unedited.
def find_root(base="/kaggle/input"):
    best, best_n = None, 0
    for dirpath, _, filenames in os.walk(base):
        n = sum(1 for f in filenames if f.endswith(".csv"))
        if n > best_n:
            best, best_n = dirpath, n
    if best is None:
        raise SystemExit(
            f"No CSVs found under {base}. Add the dataset to this notebook "
            f"with '+ Add Input' in the sidebar.")
    return best

ROOT = find_root()
files = sorted(glob.glob(os.path.join(ROOT, "*.csv")))

print(f"reading from {ROOT}")
print(f"{len(files):,} train files")
print(f"{sum(os.path.getsize(f) for f in files) / 1024**3:.1f} GB total")

In [ ]:
# One consistent look for every chart below. Blue is the single series colour;
# grid and axes stay recessive so the data reads first.
SURFACE, INK, MUTED, BLUE = "#fcfcfb", "#0b0b0b", "#52514e", "#2a78d6"
mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": "#d9d8d3", "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": "#e8e7e2", "grid.linewidth": 0.8,
    "axes.titlesize": 13, "axes.titleweight": "medium",
    "font.size": 10, "figure.dpi": 130,
})

## One train, end to end

Always read `train_no` as a string. Leading zeros are part of the number, and
pandas will happily turn `01421` into `1421` if you let it.

In [ ]:
COLUMNS = ["date", "train_no", "train_name", "train_type", "seq",
           "station_code", "halts_here", "sch_arrival", "sch_arrival_date",
           "sch_departure", "sch_departure_date", "act_arrival",
           "act_arrival_date", "act_departure", "act_departure_date",
           "delay_arrival_min", "delay_departure_min", "platform",
           "distance_km", "lat", "lng", "departed", "reported",
           "reported_at"]

DTYPE = {"train_no": str, "station_code": str, "platform": str,
         "train_name": str, "train_type": str}

# Not every file in this dataset has a header row; a minority start straight
# on data. Passing names= unconditionally would be wrong for the ones that do
# have a header, so check the first line and pick. Skipping this check costs
# you the first row of those files and gives you garbage column names.
def read(path, usecols=None):
    with open(path) as fh:
        has_header = fh.readline().startswith("date,train_no")
    kw = dict(dtype=DTYPE, parse_dates=["date"], usecols=usecols)
    if not has_header:
        kw.update(header=None, names=COLUMNS)
    return pd.read_csv(path, **kw)

def load(train_no):
    return read(os.path.join(ROOT, f"{train_no}.csv"))

headerless = sum(
    0 if open(f).readline().startswith("date,train_no") else 1
    for f in files[:500])
print(f"{headerless} of the first 500 files have no header row")

# Pick the largest file, whatever it happens to be, so this runs unedited.
biggest = max(files, key=os.path.getsize)
one = read(biggest)

print(one.train_name.mode()[0], "|", one.train_type.mode()[0])
print(f"{len(one):,} stop records, {one.date.nunique():,} runs, "
      f"{one.date.min().date()} to {one.date.max().date()}")
one.head()

## The `reported` column decides what is real

A blank `act_arrival` does not mean the train never arrived. It means that stop
never filed a status update. On a typical run about 3 stops in 23 are silent,
and it tends to be the same quiet stations every time, so dropping them is not
dropping a random sample. Everything below filters on `reported == 1` first.

In [ ]:
obs = one[one.reported == 1]
print(f"{len(obs):,} of {len(one):,} rows observed "
      f"({100 * len(obs) / len(one):.1f}%)")

# Per-run completeness: what fraction of each journey's stops reported at all.
completeness = one.groupby("date").reported.mean()
print(f"median run is {100 * completeness.median():.0f}% observed")

In [ ]:
d = obs.delay_arrival_min.dropna()
d = d[d < 300]  # the tail runs for hours; clip it so the bulk is visible

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.hist(d, bins=60, color=BLUE, edgecolor=SURFACE, linewidth=0.6)
ax.set_axisbelow(True); ax.yaxis.grid(True); ax.xaxis.grid(False)
ax.set_xlabel("Arrival delay (minutes)")
ax.set_ylabel("Stop records")
ax.set_title(f"Arrival delay at every observed stop, train {one.train_no.iloc[0]}")

med, p90 = d.median(), d.quantile(0.90)
for val, label in ((med, f"median {med:.0f}m"), (p90, f"p90 {p90:.0f}m")):
    ax.axvline(val, color=INK, linewidth=1, linestyle="--", alpha=0.55)
    ax.annotate(label, (val, ax.get_ylim()[1] * 0.92), xytext=(5, 0),
                textcoords="offset points", fontsize=9, color=INK)
plt.show()

That spike at zero is not a fleet of perfectly punctual trains. The source
reports `0` for on time **and** for early, so everything at or ahead of schedule
collapses into one bar. Treat every delay figure in this dataset as a lower
bound.

## Where the delay comes from

`distance_km` is cumulative from the origin, so pairing it with
`delay_arrival_min` gives a delay profile along the route. Averaged over every
run, it shows which stretches lose time and which ones claw it back.

In [ ]:
profile = (obs.dropna(subset=["distance_km", "delay_arrival_min"])
              .groupby(["seq", "station_code"])
              .agg(km=("distance_km", "median"),
                   delay=("delay_arrival_min", "median"),
                   n=("delay_arrival_min", "size"))
              .reset_index()
              .query("n >= 20")
              .sort_values("km"))

fig, ax = plt.subplots(figsize=(8.4, 3.6))
ax.plot(profile.km, profile.delay, color=BLUE, linewidth=2,
        marker="o", markersize=4, markeredgecolor=SURFACE, markeredgewidth=1.2)
ax.set_axisbelow(True); ax.yaxis.grid(True)
ax.set_xlabel("Distance from origin (km)")
ax.set_ylabel("Median arrival delay (min)")
ax.set_title("Delay builds along the route")

# Label only the ends and the worst stop, not every point.
worst = profile.loc[profile.delay.idxmax()]
for row in (profile.iloc[0], worst, profile.iloc[-1]):
    ax.annotate(row.station_code, (row.km, row.delay), xytext=(0, 8),
                textcoords="offset points", ha="center", fontsize=9, color=INK)
plt.show()

## Across many trains

Reading all 12,935 files takes a while. A few hundred of the largest ones is
enough to see fleet-wide behaviour, and they are the daily long distance
services rather than one-off specials.

In [ ]:
USE = ["date", "train_no", "train_type", "station_code",
       "delay_arrival_min", "lat", "lng", "reported"]

sample_files = sorted(files, key=os.path.getsize, reverse=True)[:300]
frames = []
for f in sample_files:
    df = read(f, usecols=USE)
    frames.append(df[df.reported == 1])

big = pd.concat(frames, ignore_index=True)
del frames
print(f"{len(big):,} observed stop records across {big.train_no.nunique()} trains")

In [ ]:
monthly = (big.dropna(subset=["delay_arrival_min"])
              .set_index("date").delay_arrival_min
              .resample("MS").agg(["median", "size"])
              .query("size >= 500"))

fig, ax = plt.subplots(figsize=(8.8, 3.4))
ax.plot(monthly.index, monthly["median"], color=BLUE, linewidth=2)
ax.set_axisbelow(True); ax.yaxis.grid(True)
ax.set_ylabel("Median arrival delay (min)")
ax.set_title("Monthly median arrival delay, 300 busiest trains")
ax.set_ylim(bottom=0)
plt.show()

print("Gaps in this line are real. Passenger services stopped from roughly")
print("2020-03-22 into late 2021, so there is no data to plot there.")

In [ ]:
# Every row carries coordinates, so a station map needs no geocoding step.
stations = (big.dropna(subset=["lat", "lng", "delay_arrival_min"])
               .groupby("station_code")
               .agg(lat=("lat", "median"), lng=("lng", "median"),
                    delay=("delay_arrival_min", "median"),
                    n=("delay_arrival_min", "size"))
               .query("n >= 50 and 6 < lat < 37 and 67 < lng < 98"))

fig, ax = plt.subplots(figsize=(6.6, 7.0))
sc = ax.scatter(stations.lng, stations.lat, c=stations.delay,
                cmap="Blues", vmin=0, vmax=stations.delay.quantile(0.95),
                s=9, linewidths=0)
ax.set_aspect(1.0); ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title(f"{len(stations):,} stations, by median arrival delay")
cb = fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.01)
cb.set_label("Median arrival delay (min)")
cb.outline.set_visible(False)
plt.show()

In [ ]:
by_type = (big.dropna(subset=["delay_arrival_min"])
              .groupby("train_type").delay_arrival_min
              .agg(["median", "size"])
              .query("size >= 2000")
              .sort_values("median"))

fig, ax = plt.subplots(figsize=(7.2, 0.42 * len(by_type) + 1.2))
ax.barh(by_type.index, by_type["median"], color=BLUE, height=0.62)
ax.set_axisbelow(True); ax.xaxis.grid(True); ax.yaxis.grid(False)
ax.set_xlabel("Median arrival delay (min)")
ax.set_title("Punctuality by service class")
for name, row in by_type.iterrows():
    ax.annotate(f"{row['median']:.0f}", (row["median"], name), xytext=(5, 0),
                textcoords="offset points", va="center", fontsize=9, color=INK)
plt.show()

## Where to take it next

- Predict arrival delay at stop *N* from the delays at stops 1 to *N-1*. The
  data is already in the right shape for it, one run per `(train_no, date)`.
- North Indian winter fog against southern routes, splitting on `lat`.
- Whether the late-2021 restart changed punctuality, using the shutdown as a
  natural break rather than a hole to interpolate over.
- Recovery behaviour: stretches where median delay *falls* with distance are
  places the timetable has slack built in.

One warning worth repeating. Because early running is recorded as zero, any
model trained on these delays is learning a censored variable. If that matters
for your question, a tobit or a two-part model will treat the zeros honestly.